In [ ]:
import warnings
import numpy as np
import pandas as pd
from pyprojroot import here
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss, grangercausalitytests

## Load and Preview Data
The following code loads the merged dataset and displays the first few rows.

In [ ]:
df = pd.read_csv(here("data/processed/merged_data.csv"))

df["datetime"] = pd.to_datetime(df["datetime"])

df.head()

## Stationarity

Taken from [here](https://www.statsmodels.org/stable/examples/notebooks/generated/stationarity_detrending_adf_kpss.html) (2026-03-09) for ease.

In [ ]:
def adf_test(timeseries):
    """Dickey-Fuller Test for stationarity.
    
    - p < 0.05: Reject null hypothesis of non-stationarity (stationary)
    - p >= 0.05: Fail to reject null hypothesis of non-stationarity (non-stationary)
    """
    print("Results of Dickey-Fuller Test:")
    dftest = adfuller(timeseries, autolag="AIC")
    adfoutput = pd.Series(
        dftest[0:4],
        index=[
            "Test Statistic",
            "p-value",
            "#Lags Used",
            "Number of Observations Used",
        ],
    )
    for key, value in dftest[4].items():
        adfoutput["Critical Value (%s)" % key] = value

    print(adfoutput)


def kpss_test(timeseries):
    """KPSS Test for stationarity.
    
    - p < 0.05: Reject null hypothesis of stationarity (non-stationary)
    - p >= 0.05: Fail to reject null hypothesis of stationarity (stationary)
    """
    print("Results of KPSS Test:")
    kpsstest = kpss(timeseries, regression="c", nlags="auto")
    kpss_output = pd.Series(
        kpsstest[0:3], index=["Test Statistic", "p-value", "Lags Used"]
    )
    for key, value in kpsstest[3].items():
        kpss_output["Critical Value (%s)" % key] = value

    print(kpss_output)

### Cases

In [ ]:
adf_test(df["cases"])
print("")
kpss_test(df["cases"])

**Conclusion**: Borderline, assume Non-Stationary.

### Passengers

In [ ]:
adf_test(df["passengers"])
print("")
kpss_test(df["passengers"])

**Conclusion**: Non-Stationary

### PYPL

In [ ]:
adf_test(df["PYPL"])
print("")
kpss_test(df["PYPL"])

**Conclusion**: Non-Stationary

## Granger

In [ ]:
def granger_causality(df, x, y, lag=4, max_lag=4, sig=0.05):
    """Perform Granger causality test between two time series."""

    # Suppress FutureWarnings from statsmodels
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        print(f"Testing: {x} -> {y}")
        gc_result = grangercausalitytests(df[[y, x]], maxlag=max_lag, verbose=False)
        p_forward = gc_result[lag][0]['ssr_ftest'][1]
        print(f"\tp: {p_forward:0.4f}")

        print(f"Testing: {y} -> {x}")
        gc_result_reverse = grangercausalitytests(df[[x, y]], maxlag=max_lag, verbose=False)
        p_reverse = gc_result_reverse[lag][0]['ssr_ftest'][1]
        print(f"\tp: {p_reverse:0.4f}")

    print("\nCausality Result:")
    if p_forward < sig and p_reverse < sig:
        print("Bidirectional causality")
    elif p_forward < sig and p_reverse >= sig:
        print(f"{x!r} Granger-causes {y!r} at lag {lag} and {sig:0.0%} significance level")
    elif p_reverse < sig and p_forward >= sig:
        print(f"{y!r} Granger-causes {x!r} at lag {lag} and {sig:0.0%} significance level")
    else:
        print(f"No Granger causality at lag {lag} and {sig:0.0%} significance level")

    return p_forward, p_reverse

### Cases vs Passengers

In [ ]:
forward, reverse = granger_causality(df, "cases", "passengers")

### Cases vs Stock

In [ ]:
forward, reverse = granger_causality(df, "cases", "PYPL")

### Passengers vs Stock

In [ ]:
forward, reverse = granger_causality(df, "passengers", "PYPL")